# Word Embeddings

**Course:** [Natural Language Processing](https://ml-viz.vercel.app/courses/nlp/02-word-embeddings)

This notebook builds a simplified Word2Vec skip-gram model, visualizes semantic geometry (king − man + woman ≈ queen), and shows why contextual embeddings are needed for polysemous words.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Skip-gram Word2Vec — toy implementation

Skip-gram learns embeddings by training a neural network to predict context words from center words.

In [ ]:
corpus = [
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the king and queen are royalty",
    "the man went to the market",
    "the woman went to the market",
    "the man and woman are human",
    "royalty includes king and queen",
    "human includes man and woman",
]

# Build vocabulary
tokens = ' '.join(corpus).split()
vocab = sorted(set(tokens))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}
V = len(vocab)
print(f"Vocabulary ({V} words): {vocab}")

In [ ]:
# Generate skip-gram training pairs
def generate_pairs(corpus, window=2):
    pairs = []
    for sentence in corpus:
        words = sentence.split()
        for i, center in enumerate(words):
            for j in range(max(0, i-window), min(len(words), i+window+1)):
                if i != j:
                    pairs.append((w2i[center], w2i[words[j]]))
    return pairs

pairs = generate_pairs(corpus)
print(f"Generated {len(pairs)} training pairs")
print("Sample:", [(i2w[c], i2w[ctx]) for c, ctx in pairs[:5]])

In [ ]:
# Train skip-gram with negative sampling
D = 4  # embedding dimension
W = np.random.randn(V, D) * 0.1   # input embeddings (the word vectors)
C = np.random.randn(V, D) * 0.1   # output embeddings

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -20, 20)))

def train_step(center_idx, context_idx, neg_samples, lr=0.05):
    # Positive pair gradient
    pos_score = np.dot(W[center_idx], C[context_idx])
    pos_grad = (sigmoid(pos_score) - 1)
    W[center_idx] -= lr * pos_grad * C[context_idx]
    C[context_idx] -= lr * pos_grad * W[center_idx]
    # Negative pairs gradient
    for neg_idx in neg_samples:
        neg_score = np.dot(W[center_idx], C[neg_idx])
        neg_grad = sigmoid(neg_score)
        W[center_idx] -= lr * neg_grad * C[neg_idx]
        C[neg_idx] -= lr * neg_grad * W[center_idx]

# Training loop
k = 5  # negative samples
for epoch in range(200):
    np.random.shuffle(pairs)
    for center, context in pairs:
        negs = np.random.choice([i for i in range(V) if i != context], k)
        train_step(center, context, negs, lr=0.05)

print("Training complete.")
print("\nEmbedding for 'king':", W[w2i['king']].round(3))

## Semantic geometry: king − man + woman

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def most_similar(query_vec, exclude=(), top_n=3):
    sims = [(i2w[i], cosine_sim(query_vec, W[i])) for i in range(V) if i2w[i] not in exclude]
    return sorted(sims, key=lambda x: -x[1])[:top_n]

# Analogy: king - man + woman ≈ ?
analogy = W[w2i['king']] - W[w2i['man']] + W[w2i['woman']]
result = most_similar(analogy, exclude={'king', 'man', 'woman'})
print("king − man + woman ≈", result)

In [ ]:
# PCA projection for visualization (D=4 → 2D)
from numpy.linalg import eigh

def pca2(X):
    X = X - X.mean(0)
    cov = X.T @ X / len(X)
    vals, vecs = eigh(cov)
    return X @ vecs[:, -2:]

embs_2d = pca2(W)

highlight = {'king': '#6366f1', 'queen': '#6366f1', 'man': '#2dd4bf', 'woman': '#2dd4bf',
             'royalty': '#f97316', 'human': '#f97316'}

fig, ax = plt.subplots(figsize=(8, 6))
for i, word in enumerate(vocab):
    color = highlight.get(word, '#64748b')
    ax.scatter(*embs_2d[i], color=color, s=80, zorder=3)
    ax.annotate(word, embs_2d[i], textcoords='offset points', xytext=(6, 3),
                fontsize=9, color=color)

# Draw analogy arrows
for a, b, c, d in [('man', 'king', 'woman', 'queen')]:
    va, vb = embs_2d[w2i[a]], embs_2d[w2i[b]]
    vc, vd = embs_2d[w2i[c]], embs_2d[w2i[d]]
    ax.annotate('', xy=vb, xytext=va, arrowprops=dict(arrowstyle='->', color='#6366f1', alpha=0.6))
    ax.annotate('', xy=vd, xytext=vc, arrowprops=dict(arrowstyle='->', color='#2dd4bf', alpha=0.6))

# Legend patches
legend_items = [
    mpatches.Patch(color='#6366f1', label='Royalty (king/queen)'),
    mpatches.Patch(color='#2dd4bf', label='Human (man/woman)'),
    mpatches.Patch(color='#f97316', label='Category words'),
]
ax.legend(handles=legend_items, loc='lower right', facecolor='#1a1d27', edgecolor='#2a2d3a')
ax.set_title('Word2Vec embeddings — PCA projection (king−man+woman≈queen)', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Static vs contextual embeddings: polysemy

Static embeddings assign one vector per word type. Contextual embeddings (like BERT) assign different vectors depending on sentence context.

In [ ]:
# Simulate static vs contextual with simple averaging (static) vs sentence context (contextual)
bank_sentences = [
    "She sat by the river bank watching ducks",
    "The river bank was covered in mud",
    "He opened a bank account for savings",
    "The bank approved his mortgage loan",
]

# For illustration: 'river/mud/ducks/watching' → nature context, 'account/savings/loan/mortgage' → finance
# Static embedding: always the same random vector for 'bank'
static_bank = np.array([0.3, 0.1, -0.2, 0.4])  # fixed for all contexts

# Contextual: different vector based on surrounding words (conceptual)
nature_context = np.array([0.8, 0.1, 0.7, 0.2])   # "river", "mud", "ducks" nearby
finance_context = np.array([-0.1, 0.9, -0.3, 0.8])  # "account", "loan" nearby

contextual_bank = [
    nature_context, nature_context,
    finance_context, finance_context,
]

print("Static 'bank' vector (same for all sentences):")
print(f"  {static_bank}")
print("\nContextual 'bank' vectors:")
for i, s in enumerate(bank_sentences):
    print(f"  Sentence {i+1}: {contextual_bank[i]}  | '{s[:45]}...'")

In [ ]:
# Similarity: river-bank with finance-bank
print("Static: cosine(bank_s1, bank_s3) =", cosine_sim(static_bank, static_bank).round(3),
      "(always 1.0 — same vector)")
print("Contextual: cosine(bank_nature, bank_finance) =",
      cosine_sim(nature_context, finance_context).round(3),
      "(different senses → low similarity)")

## ✏️ Your turn

### Exercise 1: Implement cosine-similarity analogy solver

Given embeddings for a, b, c, find the word d such that `a : b :: c : d` (i.e., `b - a + c ≈ d`).

In [ ]:
def solve_analogy(a, b, c, embeddings, vocab_list):
    """
    Solve analogy: a is to b as c is to ?
    Returns the word with highest cosine similarity to (b - a + c),
    excluding a, b, c themselves.
    
    Args:
        a, b, c: str, word strings
        embeddings: dict mapping word -> np.array
        vocab_list: list of all word strings
    Returns:
        str: best matching word
    """
    # TODO(you): compute query vector, find nearest neighbor excluding a, b, c
    pass


emb_dict = {w: W[w2i[w]] for w in vocab}
answer = solve_analogy('man', 'king', 'woman', emb_dict, vocab)
print(f"man : king :: woman : {answer}")

In [ ]:
answer = solve_analogy('man', 'king', 'woman', emb_dict, vocab)
assert isinstance(answer, str), "Should return a string"
assert answer not in ('man', 'king', 'woman'), "Should not return the input words"
print(f"✅ Exercise 1 passed — analogy answer: '{answer}'")

<details>
<summary>💡 Show solution</summary>

```python
def solve_analogy(a, b, c, embeddings, vocab_list):
    query = embeddings[b] - embeddings[a] + embeddings[c]
    exclude = {a, b, c}
    best_word, best_sim = None, -float('inf')
    for word in vocab_list:
        if word in exclude:
            continue
        sim = cosine_sim(query, embeddings[word])
        if sim > best_sim:
            best_sim = sim
            best_word = word
    return best_word
```
</details>

### Exercise 2: Compute a co-occurrence matrix (GloVe style)

GloVe builds a global co-occurrence matrix X where X[i][j] counts how often word j appears in the context window of word i.

In [ ]:
def build_cooccurrence(corpus_sentences, word2idx, window=2):
    """
    Build a co-occurrence count matrix.
    
    Args:
        corpus_sentences: list of str
        word2idx: dict mapping word -> int index
        window: int, context window radius
    Returns:
        np.ndarray of shape (V, V) with co-occurrence counts
    """
    V = len(word2idx)
    X = np.zeros((V, V))
    # TODO(you): for each sentence, for each center word at position i,
    # increment X[center_idx][context_idx] for all context positions within window
    return X


X = build_cooccurrence(corpus, w2i)
print(f"Co-occurrence matrix shape: {X.shape}")
print(f"king <-> queen co-occurrences: {X[w2i['king'], w2i['queen']]:.0f}")
print(f"king <-> man co-occurrences:   {X[w2i['king'], w2i['man']]:.0f}")

In [ ]:
X = build_cooccurrence(corpus, w2i)
assert X.shape == (V, V), "Wrong shape"
assert X[w2i['king'], w2i['queen']] > 0, "king and queen should co-occur"
assert X[w2i['king'], w2i['man']] >= 0, "Should have non-negative counts"
assert X[w2i['king'], w2i['king']] == 0 or True, "Self co-occurrence can be 0"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def build_cooccurrence(corpus_sentences, word2idx, window=2):
    V = len(word2idx)
    X = np.zeros((V, V))
    for sentence in corpus_sentences:
        words = sentence.split()
        for i, center in enumerate(words):
            if center not in word2idx:
                continue
            ci = word2idx[center]
            for j in range(max(0, i-window), min(len(words), i+window+1)):
                if i != j and words[j] in word2idx:
                    X[ci][word2idx[words[j]]] += 1
    return X
```
</details>